# Phân tích quỹ dự trữ - Công ty bảo hiểm Beta

**Bài toán:** Số lượng yêu cầu bồi thường đến theo quá trình Poisson $N(t)$ với cường độ $\lambda=3$ vụ/ngày. Mức bồi thường $Y_i$ độc lập, phân phối mũ với tham số trung bình $\mu=0.25$ (triệu USD). Vốn ban đầu $u=20$ triệu USD, tốc độ thu phí $c=15$ triệu USD/ngày.

**Mục tiêu:**
- Trình bày phân phối của $W_i$ và $T_4$, cùng xác suất $P(T_2<1)$.
- Tính $E[X_t]$, $\mathrm{Std}(X_t)$, $E[R_t]$, $\mathrm{Var}(R_t)$ tại $t=10$.
- Tính $P(N_1=0, S_2>5)$ và xác suất phá sản theo Cramer-Lundberg.
- Minh họa bằng mô phỏng và biểu đồ để làm rõ quá trình Poisson.

**Đơn vị:** Tiền tệ là triệu USD, thời gian tính theo ngày.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

lambda_rate = 3.0
mu_claim = 0.25
beta_rate = 1.0 / mu_claim
u = 20.0
c = 15.0
t = 10.0
threshold = 5.0

params = pd.DataFrame({
    "Parameter": ["lambda", "mu (mean claim)", "beta (rate)", "u", "c", "t", "threshold"],
    "Value": [lambda_rate, mu_claim, beta_rate, u, c, t, threshold],
    "Unit": ["events/day", "million USD", "1/million USD", "million USD", "million USD/day", "days", "million USD"],
})
params

## 1. Phân phối $W_i$ và $T_4$, xác suất $T_2 < 1$

- $W_i \sim \text{Exp}(\lambda)$ (thời gian chờ giữa 2 yêu cầu liên tiếp).
- $T_4 = \sum_{i=1}^4 W_i \sim \text{Erlang}(k=4, \text{ rate } \lambda)$.
- $P(T_2 < 1) = 1 - e^{-\lambda}(1+\lambda)$.

In [ ]:
t2_prob = 1.0 - math.exp(-lambda_rate) * (1.0 + lambda_rate)
print(f"P(T2 < 1 day) = {t2_prob:.6f}")

## 2. Kỳ vọng và độ lệch chuẩn của tổng bồi thường $X_t$

Với quá trình Poisson phức hợp:

$$E[X_t] = \lambda t E[Y], \quad Var(X_t) = \lambda t E[Y^2].$$

Với $Y \sim \text{Exp}(\mu)$ theo tham số trung bình $\mu$, ta có $E[Y]=\mu$ và $E[Y^2]=2\mu^2$.

In [ ]:
E_Y = mu_claim
E_Y2 = 2.0 * mu_claim ** 2

E_Xt = lambda_rate * t * E_Y
Var_Xt = lambda_rate * t * E_Y2
Std_Xt = math.sqrt(Var_Xt)

E_Rt = u + c * t - E_Xt
Var_Rt = Var_Xt

summary = pd.DataFrame({
    "Quantity": ["E[X_t]", "Std[X_t]", "E[R_t]", "Var[R_t]"],
    "Value": [E_Xt, Std_Xt, E_Rt, Var_Rt],
    "Unit": ["million USD", "million USD", "million USD", "(million USD)^2"],
})
summary

## 3. Xác suất $\{N_1=0, S_2 > 5\}$

Do các khoảng thời gian rời rạc độc lập: $P(N_1=0, S_2>5) = P(N_1=0) \cdot P(S_2>5)$.
Trong đó $S_2$ là tổng bồi thường trong ngày thứ hai (cửa sổ 1 ngày).

In [ ]:
def erlang_survival(rate: float, shape: int, x: float) -> float:
    if x <= 0.0:
        return 1.0
    term = 1.0
    series_sum = 1.0
    for k in range(1, shape):
        term *= rate * x / k
        series_sum += term
    return math.exp(-rate * x) * series_sum

def compound_poisson_tail(lambda_rate: float, beta_rate: float, threshold: float, time_window: float = 1.0, tol: float = 1e-12) -> float:
    mean = lambda_rate * time_window
    prob_n = math.exp(-mean)
    cumulative = prob_n
    tail_prob = 0.0
    n = 0
    while True:
        if n >= 1:
            tail_prob += prob_n * erlang_survival(beta_rate, n, threshold)
        n += 1
        prob_n *= mean / n
        cumulative += prob_n
        if 1.0 - cumulative < tol or n > 500:
            break
    return tail_prob

prob_day1_zero = math.exp(-lambda_rate)
prob_day2_exceed = compound_poisson_tail(lambda_rate, beta_rate, threshold)
prob_joint = prob_day1_zero * prob_day2_exceed

print(f"P(N1=0) = {prob_day1_zero:.6f}")
print(f"P(S2>5) = {prob_day2_exceed:.6f}")
print(f"Joint probability = {prob_joint:.6f}")

## 4. Xác suất phá sản theo Cramer-Lundberg

Với bồi thường mũ, hệ số điều chỉnh $R$ thỏa:
$$R = \beta - \frac{\lambda}{c}, \quad (c > \lambda E[Y])$$
Xác suất phá sản: $\psi(u) = e^{-R u}$.

In [ ]:
if c > lambda_rate * mu_claim:
    adjustment = beta_rate - lambda_rate / c
    ruin_prob = math.exp(-adjustment * u)
else:
    adjustment = None
    ruin_prob = 1.0

print(f"Adjustment R = {adjustment:.6f}")
print(f"Ruin probability = {ruin_prob:.6e}")

## 5. Minh họa phân phối (Monte Carlo)

Sinh mẫu bằng **Inverse Transform Sampling** để minh bạch hóa thuật toán.
Phần mô phỏng này hỗ trợ trực quan hóa $W_i$, $Y_i$, $T_4$ và $N(t)$.

In [ ]:
rng = np.random.default_rng(2026)

def sample_exp_rate(rate: float, size: int, rng: np.random.Generator) -> np.ndarray:
    u = rng.random(size)
    u = np.clip(u, 1e-12, 1.0 - 1e-12)
    return -np.log(u) / rate

def sample_erlang(rate: float, shape: int, size: int, rng: np.random.Generator) -> np.ndarray:
    total = np.zeros(size)
    for _ in range(shape):
        total += sample_exp_rate(rate, size, rng)
    return total

def sample_poisson_counts(lam: float, size: int, rng: np.random.Generator) -> np.ndarray:
    samples = []
    for _ in range(size):
        u = rng.random()
        p = math.exp(-lam)
        cumulative = p
        k = 0
        while u > cumulative:
            k += 1
            p *= lam / k
            cumulative += p
        samples.append(k)
    return np.array(samples)

In [ ]:
sample_size = 4000
waiting_times = sample_exp_rate(lambda_rate, sample_size, rng)
claim_sizes = sample_exp_rate(beta_rate, sample_size, rng)
t4_samples = sample_erlang(lambda_rate, 4, sample_size, rng)

def exp_pdf(x: np.ndarray, rate: float) -> np.ndarray:
    return rate * np.exp(-rate * x)

def erlang_pdf(x: np.ndarray, rate: float, shape: int) -> np.ndarray:
    return (rate ** shape) * (x ** (shape - 1)) * np.exp(-rate * x) / math.factorial(shape - 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

x1 = np.linspace(0, np.percentile(waiting_times, 99), 200)
axes[0].hist(waiting_times, bins=30, density=True, alpha=0.6, color="#2f80ed")
axes[0].plot(x1, exp_pdf(x1, lambda_rate), color="#1b2631", linewidth=2)
axes[0].set_title("Waiting times W_i")
axes[0].set_xlabel("Days")

x2 = np.linspace(0, np.percentile(claim_sizes, 99), 200)
axes[1].hist(claim_sizes, bins=30, density=True, alpha=0.6, color="#f2994a")
axes[1].plot(x2, exp_pdf(x2, beta_rate), color="#1b2631", linewidth=2)
axes[1].set_title("Claim sizes Y_i")
axes[1].set_xlabel("Million USD")

x3 = np.linspace(0, np.percentile(t4_samples, 99), 200)
axes[2].hist(t4_samples, bins=30, density=True, alpha=0.6, color="#27ae60")
axes[2].plot(x3, erlang_pdf(x3, lambda_rate, 4), color="#1b2631", linewidth=2)
axes[2].set_title("Event time T4")
axes[2].set_xlabel("Days")

for ax in axes:
    ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.show()

**Insight:** Các histogram khớp với đường mật độ lý thuyết; $W_i$ và $Y_i$ có đuôi dài do phân phối mũ, trong khi $T_4$ tập trung hơn vì là tổng của 4 biến mũ. Điều này phản ánh tính ổn định khi cộng dồn thời gian chờ.

In [ ]:
def poisson_pmf(lam: float, k: int) -> float:
    return math.exp(-lam) * (lam ** k) / math.factorial(k)

count_samples = sample_poisson_counts(lambda_rate * t, sample_size, rng)
max_k = int(count_samples.max())
bins = np.arange(-0.5, max_k + 1.5, 1)

fig, ax = plt.subplots(figsize=(6.2, 3.5))
ax.hist(count_samples, bins=bins, density=True, alpha=0.7, color="#2f80ed")
pmf_vals = [poisson_pmf(lambda_rate * t, k) for k in range(max_k + 1)]
ax.plot(range(max_k + 1), pmf_vals, "o-", color="#1b2631")
ax.set_title("Poisson counts N(t) vs PMF")
ax.set_xlabel(f"N(t) at t={t}")
ax.set_ylabel("Probability")
ax.grid(True, linestyle=":", linewidth=0.5)
plt.show()

print(f"Sample mean: {count_samples.mean():.3f}")
print(f"Sample var: {count_samples.var():.3f}")
print(f"Theory mean=var: {lambda_rate * t:.3f}")

**Insight:** Giá trị trung bình và phương sai mẫu gần bằng $\lambda t$, xác nhận tính chất $E[N(t)] = Var[N(t)]$ của quá trình Poisson.

In [ ]:
def simulate_path(u: float, c: float, lambda_rate: float, mean_claim: float, T: float, rng: np.random.Generator):
    time = 0.0
    total_claims = 0.0
    times = [0.0]
    surplus = [u]
    counts = [0]
    while True:
        w = sample_exp_rate(lambda_rate, 1, rng)[0]
        time += w
        if time > T:
            break
        claim = sample_exp_rate(1.0 / mean_claim, 1, rng)[0]
        total_claims += claim
        current_surplus = u + c * time - total_claims
        times.append(time)
        surplus.append(current_surplus)
        counts.append(counts[-1] + 1)
        if current_surplus < 0:
            break
    if times[-1] < T:
        times.append(T)
        surplus.append(surplus[-1])
        counts.append(counts[-1])
    return np.array(times), np.array(surplus), np.array(counts)

times, surplus, counts = simulate_path(u, c, lambda_rate, mu_claim, t, rng)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

axes[0].step(times, surplus, where="post", linewidth=2, color="#2f80ed")
axes[0].axhline(0.0, color="red", linestyle="--")
axes[0].set_title("Surplus path U(t)")
axes[0].set_xlabel("Time (days)")
axes[0].set_ylabel("Million USD")

axes[1].step(times, counts, where="post", linewidth=2, color="#f2994a")
axes[1].set_title("Counting process N(t)")
axes[1].set_xlabel("Time (days)")
axes[1].set_ylabel("Number of claims")

for ax in axes:
    ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.show()

**Insight:** Đồ thị bậc thang của $N(t)$ thể hiện rõ các bước nhảy rời rạc, còn $U(t)$ tăng tuyến tính theo phí và giảm theo bồi thường; phá sản xảy ra khi $U(t)$ cắt xuống dưới 0.

In [ ]:
mc_size = 2000
total_claims = []
reserves = []

for _ in range(mc_size):
    n = sample_poisson_counts(lambda_rate * t, 1, rng)[0]
    if n > 0:
        claims = sample_exp_rate(beta_rate, n, rng)
        total = claims.sum()
    else:
        total = 0.0
    total_claims.append(total)
    reserves.append(u + c * t - total)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].hist(total_claims, bins=30, color="#2f80ed", alpha=0.7)
axes[0].axvline(E_Xt, color="black", linestyle="--", label="E[X_t]")
axes[0].set_title("Distribution of X_t")
axes[0].set_xlabel("Million USD")
axes[0].legend()

axes[1].hist(reserves, bins=30, color="#27ae60", alpha=0.7)
axes[1].axvline(E_Rt, color="black", linestyle="--", label="E[R_t]")
axes[1].set_title("Distribution of R_t")
axes[1].set_xlabel("Million USD")
axes[1].legend()

for ax in axes:
    ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. Bảng so sánh lý thuyết vs mô phỏng (chi tiết)

Bảng dưới đây đối chiếu các đại lượng lý thuyết của quá trình Poisson và mô hình Cramer–Lundberg với ước lượng Monte Carlo, nhằm kiểm tra tính đúng đắn của giả thiết và minh hoạ sai số mô phỏng.

In [ ]:
mc_compare = 5000
rng_compare = np.random.default_rng(2027)

t2_samples = sample_exp_rate(lambda_rate, mc_compare, rng_compare) + sample_exp_rate(lambda_rate, mc_compare, rng_compare)
t2_sim = (t2_samples < 1.0).mean()

nt_counts = sample_poisson_counts(lambda_rate * t, mc_compare, rng_compare)
E_Nt_sim = nt_counts.mean()
Var_Nt_sim = nt_counts.var()

total_claims_sim = np.zeros(mc_compare)
for i, n_i in enumerate(nt_counts):
    if n_i > 0:
        total_claims_sim[i] = sample_exp_rate(beta_rate, n_i, rng_compare).sum()
reserves_sim = u + c * t - total_claims_sim

E_Xt_sim = total_claims_sim.mean()
Var_Xt_sim = total_claims_sim.var()
Std_Xt_sim = total_claims_sim.std()
E_Rt_sim = reserves_sim.mean()
Var_Rt_sim = reserves_sim.var()

n1 = sample_poisson_counts(lambda_rate, mc_compare, rng_compare)
n2 = sample_poisson_counts(lambda_rate, mc_compare, rng_compare)
s2 = np.zeros(mc_compare)
for i, n_i in enumerate(n2):
    if n_i > 0:
        s2[i] = sample_exp_rate(beta_rate, n_i, rng_compare).sum()
p_n1_zero_sim = (n1 == 0).mean()
p_s2_exceed_sim = (s2 > threshold).mean()
p_joint_sim = ((n1 == 0) & (s2 > threshold)).mean()

mc_ruin = 1000
horizon_ruin = 200.0
def simulate_ruin_once(rng: np.random.Generator) -> int:
    time = 0.0
    total = 0.0
    while True:
        time += sample_exp_rate(lambda_rate, 1, rng)[0]
        if time > horizon_ruin:
            return 0
        total += sample_exp_rate(beta_rate, 1, rng)[0]
        if u + c * time - total < 0:
            return 1

ruin_flags = [simulate_ruin_once(rng_compare) for _ in range(mc_ruin)]
ruin_sim = float(np.mean(ruin_flags))

rows = [
    {"Dai luong": "P(T2<1)", "Ly thuyet": t2_prob, "Mo phong": t2_sim},
    {"Dai luong": "E[N(t)]", "Ly thuyet": lambda_rate * t, "Mo phong": E_Nt_sim},
    {"Dai luong": "Var[N(t)]", "Ly thuyet": lambda_rate * t, "Mo phong": Var_Nt_sim},
    {"Dai luong": "E[X_t]", "Ly thuyet": E_Xt, "Mo phong": E_Xt_sim},
    {"Dai luong": "Std[X_t]", "Ly thuyet": Std_Xt, "Mo phong": Std_Xt_sim},
    {"Dai luong": "Var[X_t]", "Ly thuyet": Var_Xt, "Mo phong": Var_Xt_sim},
    {"Dai luong": "E[R_t]", "Ly thuyet": E_Rt, "Mo phong": E_Rt_sim},
    {"Dai luong": "Var[R_t]", "Ly thuyet": Var_Rt, "Mo phong": Var_Rt_sim},
    {"Dai luong": "P(N1=0)", "Ly thuyet": prob_day1_zero, "Mo phong": p_n1_zero_sim},
    {"Dai luong": "P(S2>5)", "Ly thuyet": prob_day2_exceed, "Mo phong": p_s2_exceed_sim},
    {"Dai luong": "P(N1=0, S2>5)", "Ly thuyet": prob_joint, "Mo phong": p_joint_sim},
    {"Dai luong": "Ruin probability", "Ly thuyet": ruin_prob, "Mo phong": ruin_sim},
]

df_compare = pd.DataFrame(rows)
def rel_err(theory: float, sim: float) -> float:
    if theory == 0.0:
        return float("nan")
    return (sim - theory) / theory * 100.0

df_compare["Sai lech"] = df_compare["Mo phong"] - df_compare["Ly thuyet"]
df_compare["Sai lech (%)"] = [rel_err(t, s) for t, s in zip(df_compare["Ly thuyet"], df_compare["Mo phong"])]
df_compare

**Nhận xét:** Các đại lượng dựa trên Poisson và phân phối mũ khớp rất sát giữa lý thuyết và mô phỏng (sai lệch nhỏ do Monte Carlo).
Riêng xác suất phá sản lý thuyết cực nhỏ nên mô phỏng theo chân trời hữu hạn $T=200$ ngày thường cho kết quả 0; để ước lượng chính xác cần kỹ thuật như importance sampling hoặc tăng cỡ mẫu rất lớn.

## Tổng kết chi tiết

- Phân phối $W_i$ và $T_4$ phù hợp với Exp/Erlang; $P(T_2<1)$ được tính trực tiếp từ công thức đóng.
- Kỳ vọng và độ lệch chuẩn $X_t$ thể hiện mức biến động của tổng bồi thường sau 10 ngày; phân phối lệch phải do tổng các biến mũ.
- Xác suất $P(N_1=0, S_2>5)$ cho thấy rủi ro ngày thứ hai vẫn đáng kể dù ngày đầu yên ắng.
- Với điều kiện lợi nhuận ròng $c > \lambda E[Y]$, xác suất phá sản theo Cramer-Lundberg rất nhỏ; kết quả mô phỏng củng cố nhận định này.
- Các biểu đồ $N(t)$ và histogram Poisson làm rõ cơ chế đếm sự kiện, tăng cường trực giác về quá trình Poisson.